# Ohio air quality: interactive Altair charts

**The idea:** a short version of the AQI walkthrough. It keeps only the **7 charts** that tell the story of the project, and every chart is **interactive**: you can zoom in, move around and hover to read the numbers. The data, the clues, the split and the models are the same as on the poster.

**How to run:** choose **Runtime > Run all**. When the first cell asks, upload the Excel file `AQI_Ohio_2022_2026_with_weather.xlsx`. It takes about 1 to 2 minutes, most of it for training the models. Altair 5 or newer is needed (Colab has it).

**How to use a chart**
- **Scroll** with the mouse wheel (or two fingers on a trackpad) over a chart to **zoom**.
- **Drag** the chart to **move** it.
- **Double-click** to reset it (or run its cell again).
- **Hover** over a dot, bar or line to read its numbers.
- On a **bar chart** only the number axis zooms. The names stay in place (Altair cannot zoom a list of names).
- While the mouse is over a chart, the wheel zooms the chart. Move the mouse off the chart to scroll the page.
- To keep a chart as a picture, click the three dots at its top right and choose **Save as PNG**.

**The 7 charts**
1. **Step 2:** When is the air worst? (the average AQI of each month)
2. **Step 3:** Which counties have the worst air?
3. **Step 6:** Is tomorrow's AQI close to today's?
4. **Step 7:** Which clues move with tomorrow's AQI?
5. **Step 9:** Real vs guess, day by day.
6. **Step 10:** The scoreboard of the models.
7. **Step 11:** The bad days.

**Left out on purpose:** the charts that only show how the work was done: the histograms (of the AQI and of the weather), the scaling charts, the tuning charts, the dot chart of every single model, and the extra models (Lasso, the tuned forest, XGBoost). The long walkthrough notebook still has them.

**Words you will see**
- **AQI:** the Air Quality Index, a number for how polluted the air is. Up to 50 is good, 51 to 100 is moderate, and above 100 is unhealthy (at first for sensitive groups only).
- **Clue:** a fact we know on a day (today's AQI, the weather, the date). The models use the clues to guess tomorrow's AQI.
- **Real AQI:** the AQI that was measured. **Guess:** what a model said the day before.
- **Train days / test days:** a model learns from the train days (2022-2024). We grade it on the test days (2025 to May 2026), which it has never seen.
- **Link:** how closely a clue moves together with tomorrow's AQI, from -1 to +1. A link is not a cause.
- **MAE:** how many AQI points the guess is wrong by, on average. **RMSE:** like MAE, but big mistakes count extra. Smaller is better for both.
- **Difference:** test error minus train error. A big positive Difference is the warning sign of overfitting.
- **Altair:** the Python library that draws the charts.

**Note:** the numbers in the text come from our run. If your Colab has other library versions, a number can differ in the second decimal.

## Step 1. Get the data

Run the next cell. Click **Choose files** and pick the Excel file. Each row is one county on one day: its AQI, the main pollutant and the weather.

In [ ]:
import numpy as np
import pandas as pd
import altair as alt
from google.colab import files

alt.data_transformers.disable_max_rows()              # Altair stops at 5,000 rows by default. This lifts the limit.
print("Altair version:", alt.__version__)             # these charts need Altair 5 or newer
blue, orange, grey = "#2a78d6", "#eb6834", "#555555"  # the three colors of the charts

uploaded = files.upload()                             # a "Choose files" button appears: pick the Excel file
filename = list(uploaded)[0]                          # the name of the file you uploaded
df = pd.read_excel(filename)                          # read the Excel file into a table called df
df = df.rename(columns={"county Name": "County"})     # a shorter column name
df = df.sort_values(["County", "Date"]).reset_index(drop=True)   # one county after another, oldest day first
print(len(df), "rows,", df.County.nunique(), "counties, from", df.Date.min().date(), "to", df.Date.max().date())

## Step 2. When is the air worst?

**Chart 1** draws the **average AQI of each month**: for the whole of Ohio (all counties together) and for two counties, Cuyahoga (Cleveland) and Belmont. A month is shown only when at least 20 counties sent data in it. Zoom in on one year to see it better.

In [ ]:
df["Month"] = df.Date.dt.to_period("M").dt.to_timestamp()       # the first day of the month of each day
reporting = df.groupby("Month").County.nunique()                # how many counties sent data in each month
enough = reporting[reporting >= 20]                             # keep the months in which at least 20 counties sent data
too_few = reporting[reporting < 20]                             # the other months are left out of this chart
print("Counties that sent data in a month: between", enough.min(), "and", enough.max())
print("Left out, too few counties:", [month.strftime("%b %Y") + f" ({n} counties)" for month, n in too_few.items()])

month_rows = df[df.Month.isin(enough.index)]                    # only the rows of the months we keep
ohio = month_rows.groupby("Month").AQI.mean().reset_index()     # the average of all the rows of each month
ohio["Line"] = "Ohio average"
two = month_rows[month_rows.County.isin(["Cuyahoga", "Belmont"])].groupby(["Month", "County"]).AQI.mean().reset_index()
two = two.rename(columns={"County": "Line"})                    # the two counties get the same column name
months = pd.concat([ohio, two])                                 # one table: one row per month and per line
months["AQI"] = months.AQI.round(1)
months.sample(4, random_state=0)                                # four random rows of the table

In [ ]:
line_colors = alt.Scale(domain=["Ohio average", "Cuyahoga", "Belmont"], range=[grey, orange, blue])

chart_months = alt.Chart(months).mark_line(point=True, strokeWidth=2).encode(
    x=alt.X("Month:T", title="Month"),
    y=alt.Y("AQI:Q", title="Average AQI of the month"),
    color=alt.Color("Line:N", scale=line_colors, title=None),
    tooltip=[alt.Tooltip("Month:T", format="%B %Y"), "Line:N", alt.Tooltip("AQI:Q", format=".1f")],
).properties(title="When is the air worst? Average AQI of each month", width=800, height=320).interactive()

chart_months

**What this means**
- **June is the worst month, every year.** In 2022, 2023, 2024 and 2025 the Ohio average is highest in June (57.0, 71.7, 52.4 and 55.8). It is lowest in September, October or November (35.3 to 37.6).
- **June 2023 is the highest month of the whole file (71.7).** Smoke from wildfires in Canada drifted over Ohio that month.
- Cuyahoga (orange) is above the Ohio average in every month shown, and Belmont (blue) is below it in 39 of its 42 months.
- Not every county sends data every month: between 22 and 39 do (fewer from November to February). So small steps of the grey line can come from which counties sent data.
- **May 2026 is left out.** The file has only 8 counties in that month, and 4 of them only for 1 May. Its average would give a false picture.

## Step 3. Which counties have the worst air?

**Chart 2** has one bar for each county: its average AQI over the whole file. We keep the counties with at least one year (365 days) of data, because the average of a county with very little data cannot be trusted.

In [ ]:
counts = df.groupby("County").AQI.agg(["mean", "count"])          # each county's average AQI and its number of days
long_enough = counts[counts["count"] >= 365]                      # keep the counties with at least one year of data
county_means = long_enough.reset_index().rename(columns={"mean": "Average AQI", "count": "Days"})
county_means["Average AQI"] = county_means["Average AQI"].round(1)
print(len(county_means), "counties kept. Left out:", counts[counts["count"] < 365].index.tolist())

chart_counties = alt.Chart(county_means).mark_bar(color=blue).encode(
    x=alt.X("Average AQI:Q", title="Average AQI, January 2022 to May 2026"),
    y=alt.Y("County:N", sort="-x", title=None),
    tooltip=["County:N", alt.Tooltip("Average AQI:Q", format=".1f"), alt.Tooltip("Days:Q", format=",")],
).properties(title="Average AQI of each county", width=600, height=560).interactive()

chart_counties

**What this means**
- The five highest averages are Cuyahoga (56.2), Butler (55.3), Hamilton (54.7), Franklin (53.1) and Lucas (50.3). The lowest are Belmont (29.8) and Athens (31.9). Cuyahoga is 26.4 points above Belmont.
- Four of the five highest are the counties of Cleveland (Cuyahoga), Cincinnati (Hamilton), Columbus (Franklin) and Toledo (Lucas).
- So the county matters. That is why a county's usual AQI is one of the 20 clues (Step 4).
- 36 counties have at least a year of data. Columbiana (266 days), Fayette (128) and Harrison (42) are left out. Hover over a bar to see the number of days of a county.

## Step 4. The answer and the clues

For every day we want to guess **tomorrow's AQI**. That is the **answer**. The **clues** are facts we know on that day:
- today's AQI, the AQI 1, 2 and 6 days ago, and the averages of the last 3 and the last 7 days;
- the time of year, the weekday of tomorrow, whether ozone set today's AQI, and the wind direction;
- 7 weather columns and the county's usual AQI (the last clue is added in Step 5).

Every calculation stays inside one county, so "yesterday" never comes from another county.

In [ ]:
weather = ["Temp Max (°F)", "Temp Mean (°F)", "Precipitation (in)", "Wind Speed Max (mph)",
           "Wind Gust Max (mph)", "Relative Humidity (%)", "Solar Radiation (MJ/m²)"]       # the 7 weather columns

each_county = df.groupby("County")                    # every calculation below stays inside one county

df["tomorrow"] = each_county.AQI.shift(-1)            # the ANSWER: the AQI on the next row
df["aqi_today"] = df.AQI
df["aqi_yesterday"] = each_county.AQI.shift(1)        # the AQI on the row before
df["aqi_2_days_ago"] = each_county.AQI.shift(2)
df["aqi_6_days_ago"] = each_county.AQI.shift(6)
df["aqi_3_day_average"] = each_county.AQI.transform(lambda day: day.rolling(3).mean())   # today and the 2 days before
df["aqi_7_day_average"] = each_county.AQI.transform(lambda day: day.rolling(7).mean())   # today and the 6 days before

In [ ]:
angle = 2 * np.pi * df.Date.dt.dayofyear / 365.25                  # the day of the year as an angle on a circle
df["day_of_year_sin"] = np.sin(angle)
df["day_of_year_cos"] = np.cos(angle)
df["tomorrow_weekday"] = (df.Date + pd.Timedelta(days=1)).dt.dayofweek          # 0 = Monday ... 6 = Sunday
df["ozone_main"] = (df["Defining Parameter"] == "Ozone").astype(int)            # 1 if ozone set today's AQI

wind_angle = np.deg2rad(df["Wind Direction (°)"])                  # the wind direction is also an angle on a circle
df["wind_dir_sin"] = np.sin(wind_angle)
df["wind_dir_cos"] = np.cos(wind_angle)

In [ ]:
early_clues = ["aqi_today", "aqi_yesterday", "aqi_2_days_ago", "aqi_6_days_ago", "aqi_3_day_average", "aqi_7_day_average",
               "day_of_year_sin", "day_of_year_cos", "tomorrow_weekday", "ozone_main", "wind_dir_sin", "wind_dir_cos"]
clues_without_county = early_clues + weather       # 19 clues. The 20th, county_average, needs the split of Step 5.

# Keep only the days that have every clue, and where the next row really is the next day
next_row_is_tomorrow = (each_county.Date.shift(-1) - df.Date).dt.days == 1
df = df[next_row_is_tomorrow].dropna(subset=clues_without_county + ["tomorrow"])
print("Days we can use:", len(df))

## Step 5. Train days and test days

The models learn from the **train days** (before 2025). We check them on the **test days** (2025 to May 2026), which they never see while learning. The 20th clue, the county's usual AQI, is worked out from the train days only.

In [ ]:
train = df[df.Date < "2025-01-01"].copy()        # 2022-2024: the computer learns from these days
test = df[df.Date >= "2025-01-01"].copy()        # 2025 to May 2026: we check its guesses on these days

usual = train.groupby("County").tomorrow.mean()           # each county's usual AQI, from the training days only
train["county_average"] = train.County.map(usual)
test["county_average"] = test.County.map(usual)           # the test days get the training days' numbers: no peeking

clues = early_clues + ["county_average"] + weather        # the 20 clues, in the same order as on the poster
X_train, y_train = train[clues], train.tomorrow           # X = the clues, y = the answers
X_test, y_test = test[clues], test.tomorrow
print("Training days:", len(train), "| Test days:", len(test))

## Step 6. Is tomorrow's AQI close to today's?

**Chart 3:** each dot is one county on one day. Today's AQI is along the bottom and tomorrow's AQI is up the side. On the dashed line tomorrow's AQI is the same as today's. A chart cannot draw all 33,669 train days well, so it shows 4,000 random ones. Zoom in on the crowded middle.

In [ ]:
dots = train.sample(4000, random_state=1)[["County", "Date", "aqi_today", "tomorrow"]]    # 4,000 random train days, only the 4 columns we need
top = int(max(dots.aqi_today.max(), dots.tomorrow.max()))                  # the highest AQI on the chart
same_line = pd.DataFrame({"x": [0, top], "y": [0, top]})                   # the line where tomorrow = today

dots_layer = alt.Chart(dots).mark_circle(size=20, opacity=0.35, color=blue).encode(
    x=alt.X("aqi_today:Q", title="AQI today"), y=alt.Y("tomorrow:Q", title="AQI tomorrow"),
    tooltip=["County:N", alt.Tooltip("Date:T", title="Today's date", format="%b %d, %Y"),
             alt.Tooltip("aqi_today:Q", title="AQI today"), alt.Tooltip("tomorrow:Q", title="AQI tomorrow")])
line_layer = alt.Chart(same_line).mark_line(color="black", strokeDash=[5, 4]).encode(
    x=alt.X("x:Q", title="AQI today"), y=alt.Y("y:Q", title="AQI tomorrow"))

chart_today = (dots_layer + line_layer).properties(title="Is tomorrow's AQI close to today's?", width=520, height=520).interactive()
chart_today

In [ ]:
print("Link between today's AQI and tomorrow's AQI (train days):", round(train.aqi_today.corr(train.tomorrow), 2))
near_50 = train[train.aqi_today.between(45, 55)].tomorrow                  # tomorrow's AQI after the days with a today's AQI of 45 to 55
low, high = int(near_50.quantile(0.1)), int(near_50.quantile(0.9))         # the middle 80% of those days
print(f"{len(near_50):,} days had a today's AQI of 45 to 55. On 80% of them tomorrow's AQI was between {low} and {high}.")

**What this means**
- The dots follow the dashed line: when today's AQI is high, tomorrow's AQI usually is high too. The link is 0.63 on the train days.
- But the cloud is wide. On 8,410 days today's AQI was between 45 and 55, and on 80% of them tomorrow's AQI was anywhere between 33 and 64. So today's AQI is a strong clue, but it is not enough alone.
- Guessing "tomorrow = today" (the dashed line) is the lazy **baseline**. The models have to beat it (see Chart 6 in Step 10).

## Step 7. Which clues move with tomorrow's AQI?

**Chart 4:** for each of the 20 clues we measure the **link** with tomorrow's AQI on the train days: +1 means they move up and down together exactly, 0 means no link, -1 means one goes up when the other goes down. **A bar shows a link, not a cause.** Two things can move together because something else, like the time of year, moves both.

In [ ]:
links = pd.DataFrame({"Clue": clues, "Link": [train[clue].corr(train.tomorrow) for clue in clues]})     # one number for each clue
links["Direction"] = np.where(links.Link > 0, "goes up with tomorrow's AQI", "goes down with tomorrow's AQI")
links["Link"] = links.Link.round(2)

direction_colors = alt.Scale(domain=["goes up with tomorrow's AQI", "goes down with tomorrow's AQI"], range=[orange, blue])
chart_clues = alt.Chart(links).mark_bar().encode(
    x=alt.X("Link:Q", title="Link with tomorrow's AQI (train days)"),
    y=alt.Y("Clue:N", sort="-x", title=None),
    color=alt.Color("Direction:N", scale=direction_colors, title=None),
    tooltip=["Clue:N", "Link:Q", "Direction:N"],
).properties(title="Which clues move with tomorrow's AQI?", width=600, height=440).interactive()

chart_clues

**Check: how much of a link is just the time of year?** The next cell works out the links of the weather clues again, after taking away the average of each month of the year (all Junes together, all Octobers together, and so on). What is left is the link between days of the same month.

In [ ]:
weather_clues = ["Temp Max (°F)", "Temp Mean (°F)", "Solar Radiation (MJ/m²)", "Wind Gust Max (mph)", "Relative Humidity (%)", "Precipitation (in)"]
month_of_year = train.Date.dt.month                           # 1 = January ... 12 = December
same_month = {}
for clue in weather_clues:
    clue_left = train[clue] - train.groupby(month_of_year)[clue].transform("mean")                  # the clue minus the average of its month of the year
    answer_left = train.tomorrow - train.groupby(month_of_year).tomorrow.transform("mean")          # the answer minus the average of its month of the year
    same_month[clue] = round(clue_left.corr(answer_left), 2)

pd.DataFrame({"Link (all days)": links.set_index("Clue").Link, "Link (days of the same month)": pd.Series(same_month)}).dropna()

**What this means**
- The strongest clues say how polluted the air already is: today's AQI (0.63), the 3-day average (0.54) and the 7-day average (0.51).
- The weather clues are weaker. More sunshine (0.36) and warmer days (0.21 and 0.18) go with a higher AQI tomorrow. More wind (-0.22 and -0.26), more humidity (-0.21) and more rain (-0.16) go with a lower AQI tomorrow.
- **A link is not a cause.** In the table above, the links of the temperature (0.21 and 0.18) are mostly the time of year: for days of the same month they drop to 0.08 and 0.02. The links of wind gusts (-0.27), humidity (-0.20) and rain (-0.19) stay about the same, and the link of sunshine stays clear (0.27). A link only says that two things move together.

## Step 8. Train the models and let them guess

Four models learn from the train days only. Then each one guesses every day, train days and test days. We also keep the lazy **baseline**, "tomorrow = today", which learns nothing.
- **Ridge:** a straight-line model. The clues are first put on the same scale, with the scale learned from the train days only.
- **Random forest A:** 100 trees with the default settings. Each tree may grow as deep as it likes.
- **Random forest B:** the poster's forest: 300 trees, at most 12 questions deep, at least 10 days in every leaf, and each question looks at half of the clues.
- **Hist Gradient Boosting:** trees built one after another, each one correcting the mistakes of the trees before it.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

scaler = StandardScaler().fit(X_train)                        # learn the scale from the train days only
ridge = Ridge(alpha=1.0).fit(scaler.transform(X_train), y_train)
train["Ridge"] = ridge.predict(scaler.transform(X_train))     # a guess for every train day
test["Ridge"] = ridge.predict(scaler.transform(X_test))       # a guess for every test day
print("Ridge done")

In [ ]:
trees = {"Random forest A": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
         "Random forest B": RandomForestRegressor(n_estimators=300, max_depth=12, min_samples_leaf=10, max_features=0.5, random_state=42, n_jobs=-1),
         "Hist Gradient Boosting": HistGradientBoostingRegressor(random_state=42)}
for name, model in trees.items():
    model.fit(X_train, y_train)                               # learn from the train days only
    train[name] = model.predict(X_train)                      # a guess for every train day
    test[name] = model.predict(X_test)                        # a guess for every test day (days it has never seen)
    print(name, "done")

train["Baseline"], test["Baseline"] = train.aqi_today, test.aqi_today     # the lazy guess: tomorrow = today

## Step 9. Real vs guess, day by day

**Chart 5** draws the real AQI (blue) and forest B's guess (orange) for one county, every day of the file. The **dashed** part is the train days, which the forest has seen. The **solid** part is the test days, which it has never seen. The black dashed line is where the test days start. Zoom in on a sharp up-and-down and hover over the lines to read the numbers.

A guess is made on one day for the next day, so each row is moved one day forward: the chart shows the day the guess is **about**. To see another county, change its name in the first cell below (any county of Chart 2 works), then run that cell and the three cells after it again.

In [ ]:
train["split"], test["split"] = "Train", "Test"                   # a label for every day
both = pd.concat([train, test])                                   # all days in one table
both["Day"] = both.Date + pd.Timedelta(days=1)                    # a guess is made on a day, but it is about the NEXT day

county = "Cuyahoga"                                               # change this name to see another county
one = both[both.County == county]
long = one[["Day", "split", "tomorrow", "Random forest B"]].melt(id_vars=["Day", "split"], var_name="key", value_name="AQI")
long["key"] = long["key"].map({"tomorrow": "Real AQI", "Random forest B": "Forest B's guess"})      # nicer names for the legend
print(county, "has", len(one), "days, from", one.Day.min().date(), "to", one.Day.max().date())

In [ ]:
line_colors = alt.Scale(domain=["Real AQI", "Forest B's guess"], range=[blue, orange])      # blue = real, orange = guess
line_dashes = alt.Scale(domain=["Train", "Test"], range=[[5, 3], [1, 0]])                  # train days: dashed, test days: solid

lines = alt.Chart(long).mark_line(strokeWidth=1).encode(
    x=alt.X("Day:T", title="Day being guessed"), y=alt.Y("AQI:Q", title="AQI"),
    color=alt.Color("key:N", scale=line_colors, title=None),                  # one line for the real AQI, one for the guess
    strokeDash=alt.StrokeDash("split:N", scale=line_dashes, title=None))      # dashed on the train days, solid on the test days

hover = alt.Chart(long).mark_circle(size=80, opacity=0).encode(               # invisible dots: they only show the numbers when you hover
    x=alt.X("Day:T", title="Day being guessed"), y=alt.Y("AQI:Q", title="AQI"),
    tooltip=[alt.Tooltip("Day:T", format="%b %d, %Y"), alt.Tooltip("key:N", title="Line"), "AQI:Q", alt.Tooltip("split:N", title="Days")])

In [ ]:
split_day = one[one.split == "Test"].Day.min()                                # the first test day
split_line = alt.Chart(pd.DataFrame({"Day": [split_day]})).mark_rule(color="black", strokeDash=[5, 4]).encode(
    x=alt.X("Day:T", title="Day being guessed"))

chart_days = (lines + hover + split_line).properties(title=f"Next-day AQI in {county} County: real vs forest B's guess", width=800, height=320).interactive()
chart_days

In [ ]:
print("Test days, all counties. Average distance, in AQI points, between:")
print("  forest B's guess and TODAY's AQI:     ", round((test["Random forest B"] - test.aqi_today).abs().mean(), 2))
print("  forest B's guess and the REAL answer: ", round((test["Random forest B"] - test.tomorrow).abs().mean(), 2))
print(f"{county}, test days: the highest real AQI is", int(one[one.split == "Test"].tomorrow.max()), "and the highest guess is", int(one[one.split == "Test"]["Random forest B"].max()))

**What this means** (for Cuyahoga County)
- Blue (real) and orange (guess) move together, so the forest has learned the ups and downs. But the orange line is smoother, and it leans on today's AQI: on the test days, all counties together, the guess is on average 6.25 points away from today's AQI but 8.56 away from the real answer. So a sudden jump shows up in the guess only a day later.
- The forest does not see the big spikes coming. In Cuyahoga the highest real AQI on the test days is 150 and the highest guess is 89.
- The last blue point (1 May 2026) drops to 5. That is an oddity of the file: on that day Cuyahoga, Hamilton and Preble are listed with a tiny AQI that comes from carbon monoxide (CO). It affects only 3 of the 14,596 test days.

## Step 10. The scoreboard

**Chart 6** shows, for each model, its error (RMSE) on the train days and on the test days. A model that is much better on the train days than on the test days has memorized the train days: that is **overfitting**. The table has the numbers, with the **Difference** (test minus train).

In [ ]:
rows = []
for name in ["Baseline", "Ridge", "Random forest A", "Random forest B", "Hist Gradient Boosting"]:
    rows.append({"Model": name,
                 "Train MAE": mean_absolute_error(train.tomorrow, train[name]), "Test MAE": mean_absolute_error(test.tomorrow, test[name]),
                 "Train RMSE": mean_squared_error(train.tomorrow, train[name]) ** 0.5, "Test RMSE": mean_squared_error(test.tomorrow, test[name]) ** 0.5})
scores = pd.DataFrame(rows).set_index("Model")
scores["MAE Difference"] = scores["Test MAE"] - scores["Train MAE"]          # test minus train
scores["RMSE Difference"] = scores["Test RMSE"] - scores["Train RMSE"]
scores.round(2)

In [ ]:
long_scores = scores.reset_index().melt(id_vars="Model", value_vars=["Train RMSE", "Test RMSE"], var_name="Days", value_name="RMSE")
long_scores["Days"] = long_scores.Days.map({"Train RMSE": "Train days", "Test RMSE": "Test days"})
long_scores["RMSE"] = long_scores.RMSE.round(2)

chart_scores = alt.Chart(long_scores).mark_bar().encode(
    x=alt.X("RMSE:Q", title="RMSE (smaller is better)"),
    y=alt.Y("Model:N", sort=list(scores.index), title=None),
    yOffset=alt.YOffset("Days:N", sort=["Train days", "Test days"]),
    color=alt.Color("Days:N", scale=alt.Scale(domain=["Train days", "Test days"], range=[blue, orange]), title="Error on"),
    tooltip=["Model:N", "Days:N", "RMSE:Q"],
).properties(title="Error on the train days and on the test days", width=600, height=320).interactive()

chart_scores

**What this means**
- All four models beat the lazy baseline on the test days: RMSE 12.03 to 12.30 against 14.57. Forest B, the model of the poster, is the best: its errors are about 17% smaller than the baseline's (RMSE 12.03 against 14.57, MAE 8.56 against 10.27).
- The four models are close to each other on the test days (RMSE 12.03 to 12.30).
- **Forest A has memorized the train days.** Its RMSE is 3.76 on the train days and 12.17 on the test days: a Difference of +8.41. Forest B's Difference is +2.68 and Hist Gradient Boosting's is +2.85. A big Difference is the warning sign, even when the test error looks fine.
- The baseline and Ridge have a small negative Difference. They cannot memorize the train days, and the test days were a little calmer: the average day-to-day change (the Baseline's MAE) was 10.58 on the train days and 10.27 on the test days.

## Step 11. The bad days

**Chart 7:** each dot is one test day. The real AQI is along the bottom and forest B's guess is up the side. On the dashed line the guess is perfect. **Orange dots are the days with a real AQI above 100**, the days that matter most for health. The chart shows 3,000 random test days up to 100, plus **every** test day above 100. Zoom in to see where the orange dots land.

In [ ]:
up_to_100 = test[test.tomorrow <= 100].sample(3000, random_state=1)        # 3,000 random test days with a real AQI up to 100
above_100 = test[test.tomorrow > 100]                                      # every test day with a real AQI above 100
test_dots = pd.concat([up_to_100, above_100]).copy()
test_dots["Day"] = test_dots.Date + pd.Timedelta(days=1)                   # the day the guess is about
test_dots["guess"] = test_dots["Random forest B"]
test_dots["group"] = np.where(test_dots.tomorrow > 100, "Above 100", "Up to 100")
test_dots = test_dots[["County", "Day", "tomorrow", "guess", "group"]]      # keep only the columns the chart needs
print("Test days with a real AQI above 100:", len(above_100))
print("Forest B guessed above 100 on", int((above_100["Random forest B"] > 100).sum()), "of them")

In [ ]:
top = int(max(test_dots.tomorrow.max(), test_dots.guess.max()))            # the highest AQI on the chart
perfect = pd.DataFrame({"x": [0, top], "y": [0, top]})                     # the line where the guess is perfect
dot_colors = alt.Scale(domain=["Up to 100", "Above 100"], range=[blue, orange])

dots_b = alt.Chart(test_dots).mark_circle(size=20, opacity=0.4).encode(
    x=alt.X("tomorrow:Q", title="Real AQI"), y=alt.Y("guess:Q", title="Forest B's guess"),
    color=alt.Color("group:N", scale=dot_colors, title="Real AQI"),
    tooltip=["County:N", alt.Tooltip("Day:T", format="%b %d, %Y"), alt.Tooltip("tomorrow:Q", title="Real AQI"),
             alt.Tooltip("guess:Q", title="Forest B's guess", format=".0f")])
perfect_line = alt.Chart(perfect).mark_line(color="black", strokeDash=[5, 4]).encode(
    x=alt.X("x:Q", title="Real AQI"), y=alt.Y("y:Q", title="Forest B's guess"))

chart_bad = (dots_b + perfect_line).properties(title="Real AQI vs forest B's guess on the test days", width=520, height=520).interactive()
chart_bad

The next table groups the test days by how bad the **real** AQI was. For each group it shows the average real AQI, the average guess of forest B, and the difference (guess minus real). A plus means the guess was too high, a minus means too low.

In [ ]:
bands = pd.cut(test.tomorrow, [-1, 50, 100, 1000], labels=["0-50 (good)", "51-100 (moderate)", "Above 100"])     # how bad the real day was
average = pd.DataFrame({"Days": bands.value_counts().sort_index(),
                        "Real AQI": test.tomorrow.groupby(bands, observed=True).mean(),
                        "Forest B's guess": test["Random forest B"].groupby(bands, observed=True).mean()})
average["Guess minus real"] = average["Forest B's guess"] - average["Real AQI"]
average.round(1)

The last table shows how far off each model is on average (MAE) on each kind of day. The baseline is the lazy "tomorrow = today".

In [ ]:
miss = {name: (test[name] - test.tomorrow).abs().groupby(bands, observed=True).mean() for name in scores.index}
bad_days = pd.DataFrame(miss).T                                            # one row for each model, one column for each kind of day
bad_days.round(1)

**What this means**
- On ordinary days the guesses are good: forest B is off by 6.9 points on the days up to 50 and by 10.9 on the days from 51 to 100.
- **On the bad days it fails.** Among the test days, 109 have a real AQI above 100 (less than 1 day in 100). Forest B guessed above 100 on none of them: the real AQI averaged 118.2 and its guess 67.0.
- Forest B pulls its guesses toward the middle: a little too high on the good days (+4.4), too low on the days from 51 to 100 (-9.1) and far too low on the days above 100 (-51.2). The other models do the same: on the bad days their errors are 49.0 to 51.3.
- The lazy baseline is closer on the bad days (MAE 40.1 against 51.2 for forest B) because it copies today's AQI. It is worse on the other kinds of days (8.8 and 12.6 against 6.9 and 10.9).
- So the models are good for everyday guesses, but they cannot warn about the unhealthy days.

# Summary

- **When and where:** the Ohio average AQI is highest in June in every year from 2022 to 2025. The county matters: Cuyahoga averages 56.2 and Belmont 29.8.
- **Best clues:** today's AQI (link 0.63) and the 3-day and 7-day averages (0.54 and 0.51). The weather links are weaker, and the link of the temperature is mostly the time of year.
- **Forest B beats "tomorrow = today":** on the 14,596 test days it is wrong by 8.56 AQI points on average (MAE), against 10.27 for the baseline. That is about 17% smaller.
- **Overfitting:** forest A memorized the train days (RMSE Difference +8.41). The limits on forest B keep its Difference at +2.68.
- **The weak spot is the unhealthy days:** forest B guessed above 100 on none of the 109 test days with a real AQI above 100 (the real AQI averaged 118.2, its guess 67.0).
- **Using the charts:** scroll to zoom, drag to move, double-click to reset, hover to read the numbers.